# PCAP Network Analysis: Inter-Arrival Time (IAT) and Packet Size

This notebook analyzes network packets from a `.pcap` file, specifically looking at a targeted traffic flow between a Source IP and a Destination IP. 

It calculates statistics and generates Histogram and CDF (Cumulative Distribution Function) plots for:
1. **Inter-Arrival Time (IAT):** Time delta between consecutive packets.
2. **Packet Size:** The size of the packets in bytes.

### Instructions for Google Colab:
1. Upload your `.pcap` file to the Colab environment.
2. Run the first cell to install dependencies.
3. Set your parameters in the Configuration cell.
4. Run the rest of the cells to generate the stats and plots.

## 1. Install Dependencies

In [ ]:
!pip install scapy pandas matplotlib numpy

## 2. Configuration & PCAP Parsing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scapy.all import rdpcap, IP

# ==========================================
# CONFIGURATION: Update these values!
# ==========================================
PCAP_FILE = 'capture.pcap'      # Replace with your uploaded file name
SRC_IP = '192.168.1.100'        # Replace with your Source IP
DST_IP = '192.168.1.200'        # Replace with your Destination IP
# ==========================================

print(f"Reading packet capture: {PCAP_FILE}...")
packets = rdpcap(PCAP_FILE)

timestamps = []
sizes = []

# Extract matching packets
for pkt in packets:
    if IP in pkt:
        if pkt[IP].src == SRC_IP and pkt[IP].dst == DST_IP:
            timestamps.append(float(pkt.time))
            sizes.append(len(pkt))

print(f"Found {len(timestamps)} packets matching the flow {SRC_IP} -> {DST_IP}.")

if len(timestamps) >= 2:
    # Sort timestamps in case PCAP was slightly out of order
    timestamps.sort()
    
    # Calculate IAT (Inter-Arrival Time) in milliseconds
    iats_ms = [(timestamps[i] - timestamps[i-1]) * 1000.0 for i in range(1, len(timestamps))]
else:
    print("❌ Not enough packets found to calculate Inter-Arrival Time (need at least 2).")

## 3. Generate Statistics Tables

In [ ]:
def print_statistics(data, title, unit):
    """
    Calculates and prints statistical values in a tabular format.
    """
    if len(data) == 0:
        return
        
    # Calculate statistics
    stats = {
        "Count": len(data),
        f"Mean ({unit})": np.mean(data),
        f"Std Dev ({unit})": np.std(data),
        f"Min ({unit})": np.min(data),
        f"25th %ile ({unit})": np.percentile(data, 25),
        f"Median / 50th %ile ({unit})": np.median(data),
        f"75th %ile ({unit})": np.percentile(data, 75),
        f"90th %ile ({unit})": np.percentile(data, 90),
        f"95th %ile ({unit})": np.percentile(data, 95),
        f"99th %ile ({unit})": np.percentile(data, 99),
        f"Max ({unit})": np.max(data)
    }
    
    print(f"\n{'='*40}")
    print(f"{title.upper()} STATISTICS")
    print(f"{'='*40}")
    
    stats_df = pd.DataFrame(list(stats.items()), columns=['Metric', 'Value'])
    
    def format_val(x):
        if isinstance(x, float):
            return f"{x:.4f}"
        return str(x)
        
    stats_df['Value'] = stats_df['Value'].apply(format_val)
    print(stats_df.to_string(index=False, justify='left'))
    print(f"{'='*40}\n")

if 'iats_ms' in locals():
    print_statistics(iats_ms, "Inter-Arrival Time (IAT)", "ms")
    print_statistics(sizes, "Packet Size", "Bytes")

## 4. Generate Plots (Histograms & CDFs)

In [ ]:
def plot_metrics(data, title, xlabel, filename):
    """
    Generates a figure inline with both a Histogram and a CDF plot.
    """
    data = np.array(data)
    if len(data) == 0:
        print(f"Warning: No data available for {title}. Skipping plots.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Histogram Plot
    axes[0].hist(data, bins=50, color='skyblue', edgecolor='black')
    axes[0].set_title(f'Histogram: {title}')
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Frequency (Number of Packets)')
    axes[0].grid(axis='y', alpha=0.75)

    # 2. CDF Plot
    data_sorted = np.sort(data)
    p = 1. * np.arange(len(data)) / (len(data) - 1)

    axes[1].plot(data_sorted, p, color='blue', linewidth=2)
    axes[1].set_title(f'CDF: {title}')
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel('CDF (Probability)')
    axes[1].grid(True)
    axes[1].fill_between(data_sorted, p, color='blue', alpha=0.1)

    plt.tight_layout()
    plt.show()

if 'iats_ms' in locals():
    plot_metrics(iats_ms, "Inter-Arrival Time (IAT)", "Time (Milliseconds)", "iat_plots.png")
    plot_metrics(sizes, "Packet Size", "Size (Bytes)", "packet_size_plots.png")